<a href="https://colab.research.google.com/github/S-J-I/Seth_INFO4670_Fall2026/blob/main/Week5_INFO4670_Assignment2_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [129]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [130]:
n_missing_study =  students["study_hours_reported"].isna().sum()
n_missing_study

np.int64(255)

**Your justification (1–2 sentences):** I chose to leave the missing values because there's no way of knowing the actual study time for those students. Filling them in could distort the data which I want to avoid.

In [131]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [132]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
students = students.copy()

cleanmap = {"on-campus":"On Campus", "off-campus":"Off Campus", "off campus":"Off Campus", "with family":"With Family"}

students["housing_clean"] = students["housing"].str.strip().str.lower().map(cleanmap)

In [133]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off Campus': 945, 'On Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [134]:
# TODO
impossible_ages = sorted(students.loc[(students["age"] < 15) | (students["age"] > 90), "age"].unique())

n_neg_work = (students["work_hours_per_week"] < 0).sum()

students.loc[(students["age"] < 15) | (students["age"] > 90), "age"] = np.nan
students.loc[students["work_hours_per_week"] < 0, "work_hours_per_week"] = np.nan

In [135]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [136]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset="student_id")

**Why not de-dupe enroll / activity? (1 sentence):** A student can take multiple courses & have activity reported for more than a week.

In [137]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [138]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = students_dedup.copy()
analysis["sid"] = students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int)
activity_summary = activity.groupby("student_id")["minutes_active"].sum().reset_index()
analysis = analysis.merge(activity_summary, on="student_id", how="left")

In [139]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [140]:
# TODO
matched = enroll["sid"].isin(set(analysis["sid"])).sum()
print("Matced enrollment rows:", matched)

Matced enrollment rows: 8041


In [141]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [152]:
# TODO
dates_parsed = pd.to_datetime(analysis["enrollment_date"], format="mixed", errors="coerce")
analysis["enrollment_date"] = dates_parsed

In [153]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [154]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
x = analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()
analysis["study_z"] = x / analysis["study_hours_reported"].std(ddof=1)

analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3,4], labels=["0-1", "1-2", "2-3", "3-4"])

In [155]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [156]:
# TODO:
analysis.to_csv("northgate_clean.csv", index=False)

In [157]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing study hours left blank→Didn't want to distort data by median fill
- Cleaned housing categories→Various spellings of 3 categories
- Cleaned age ranges→Ages less than 15 and more than 90 aren't feasible
- Cleaned negative work hours→Replaced with missing because impossible
- Cleaned duplicate students→Removed repeating student ids for 1 row = 1 student
- Took off NU- on student key→Made a number to compare with course_enrollments
- Didn't delete wrong age/work hours→Left other data since could be useful
- Summarized weekly activity→Kept for use later, keeps one row per student
- Mixed enrollment dates→Make sure valid dates aren't lost
- Study hours z score→Standardizes study hours to measure along the average
- GPA bands→GPA splits into four ranges

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [163]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
#Cleaning the data didn't have a signifigant impact on GPA but it did remove duplicates.
print("GPA Before:")
print(students["final_gpa"].mean())
print("")

print("GPA After:")
print(analysis["final_gpa"].mean())

print(analysis[["final_gpa", "study_hours_reported"]].corr().round(2))

GPA Before:
2.307474099654662

GPA After:
2.3053600000000003
                      final_gpa  study_hours_reported
final_gpa                  1.00                  0.69
study_hours_reported       0.69                  1.00


In [164]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
